In [ ]:
# Load Large Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/ecommerce_orders_large.csv")

df.head()



In [ ]:
# Data Type Fix

df['order_date'] = pd.to_datetime(df['order_date'])

In [ ]:
# Revenue
df['revenue'] = np.multiply(df['quantity'], df['price'])

# Total Cost
df['total_cost'] = np.multiply(df['quantity'], df['cost'])

# Net Revenue
df['net_revenue'] = np.subtract(df['revenue'], df['discount'])

# Profit
df['profit'] = np.subtract(df['net_revenue'], df['total_cost'])

# Profit Margin
df['profit_margin'] = np.divide(df['profit'], df['net_revenue'])

df[['revenue', 'total_cost', 'net_revenue', 'profit', 'profit_margin']].describe()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Inspect row with missing values
df[df['cost'].isnull()].head()

In [ ]:
# # Filling Missing Cost

cost_map = {
    "Laptop": 48000,
    "Mobile": 18000,
    "Tablet": 14000,
    "Headphone": 1500,
    "Office Chair": 4500,
    "Table": 6500
}


df['cost'] = df['cost'].fillna(df['product'].map(cost_map))

df['cost'].isnull().sum()


In [ ]:
df[['revenue', 'total_cost', 'net_revenue', 'profit', 'profit_margin']].describe()

In [ ]:
# Margin is valid only if net_revenue > 0

df['profit_margin'] = np.where(
    df['net_revenue'] > 0,
    df['profit'] / df['net_revenue'],
    np.nan
)

In [ ]:
# Cap Extreme Margins

df['profit_margin'] = df['profit_margin'].clip(-1,1)
df['profit_margin'].describe()


In [ ]:
# Overall Business Performance

total_revenue = df['net_revenue'].sum()
total_profit = df['profit'].sum()
avg_margin = df['profit_margin'].mean()

total_revenue, total_profit, avg_margin


In [ ]:
# Product-wise Performance Summary

product_summary = df.groupby('product').agg(
    total_orders=(' order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    total_cost=('cost', 'sum'),
    total_profit=('profit', 'sum'),
    avg_profit_margin=('profit_margin', 'mean')
).reset_index()

product_summary.sort_values(by='total_profit', ascending=False)


In [ ]:
# Identify Loss-Making Products

product_summary[product_summary['total_profit'] < 10]

In [ ]:
# Total Profit by Product

import matplotlib.pyplot as plt

plt.figure()
plt.bar(product_summary['product'], product_summary['total_profit'])
plt.xticks(rotation=45)
plt.title('Total Profit by Product')
plt.xlabel('Product')
plt.ylabel('Profit')
plt.tight_layout()
plt.show()


In [ ]:
# Average Profit Margin by Product

plt.figure()
plt.bar(product_summary['product'], product_summary['avg_profit_margin'])
plt.xticks(rotation=45)
plt.title('Average Profit Margin by Product')
plt.xlabel('Product')
plt.ylabel('Profit Margin')
plt.tight_layout()
plt.show()


Key Insights:

Laptops generate highest absolute profit but have moderate margins.

Headphones have high margins but low ticket size.

Some products show negative profitability due to aggressive discounting.

In [ ]:
# Understand Customer Data Structure

df.columns

In [ ]:
# Orders per Customer

customer_orders = df.groupby('customer_id')[' order_id'].count().reset_index()
customer_orders.columns = ['customer_id', 'total_orders']

customer_orders['customer_type'] = np.where(
    customer_orders['total_orders'] == 1,
    'One-time',
    'Repeat'
)

customer_orders['customer_type'].value_counts(normalize=True) * 100

In [ ]:
# Revenue & Profit by Customer

customer_value = df.groupby('customer_id').agg(
    total_orders = (' order_id', 'count'),
    total_revenue = ('revenue', 'sum'),
    total_profit = ('profit', 'sum')
).reset_index()

In [ ]:
# High Value Customers

profit_threshold = customer_value['total_profit'].quantile(0.80)

customer_value['customer_segment'] = np.where(
    customer_value['total_profit'] >= profit_threshold,
    'High Value',
    'Regular'
)

customer_value.groupby('customer_segment')[['total_revenue','total_profit']].sum()

In [ ]:
# Visualization: Profit by Customer Segment

segment_summary = customer_value.groupby('customer_segment')['total_profit'].sum()

plt.figure()
segment_summary.plot(kind='bar')
plt.title('Total Profit by Customer Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Total Profit')
plt.tight_layout()
plt.show()

Customer Insights

Repeat customers generate significantly higher profit per  user.

Top 20% customers drive majority of overall profit.

Retention strategies should prioritize high-value customers.

In [ ]:
# CUSTOMER LIFETIME VALUE (CLV) & COHORT ANALYSIS

df['order_date'] = pd.to_datetime(df['order_date'])

clv = df.groupby('customer_id').agg(
    first_purchase = ('order_date', 'min'),
    last_purchase = ('order_date', 'max'),
    total_orders = (' order_id', 'count'),
    total_revenue =('revenue', 'sum'),
    total_profit = ('profit', 'sum')   
).reset_index()

clv['customer_lifetime_days'] = (
    clv['last_purchase'] - clv['first_purchase']
).dt.days

clv[['total_profit', 'customer_lifetime_days']].describe()

In [ ]:
# CLV Segmentation

clv['clv_segment'] = pd.qcut(
    clv['total_profit'],
    q=3,
    labels = ['Low CLV', 'Mid CLV', 'High CLV']
)

clv.groupby('clv_segment')['total_profit'].sum()

In [ ]:
# Cohort Analysis

df['order_month'] = df['order_date'].dt.to_period('M')
df['cohort_month'] = df.groupby('customer_id')['order_month'].transform('min')

df['cohort_index'] =(
    (df['order_month'].dt.year - df['cohort_month'].dt.year) *12 +
    (df['order_month'].dt.month - df['cohort_month'].dt.month) + 1
)

In [ ]:
# Cohort Table

cohort_data = df.groupby(
    ['cohort_month','cohort_index']
)['customer_id'].nunique().reset_index()

cohort_pivot = cohort_data.pivot(
    index='cohort_month',
    columns='cohort_index',
    values='customer_id'
)

cohort_size = cohort_pivot.iloc[:,0]
retention = cohort_pivot.divide(cohort_size, axis=0)

retention.round(2)


In [ ]:
plt.figure()
plt.imshow(retention, aspect='auto')
plt.colorbar()
plt.title('Customer Retention Cohort')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort Month')
plt.show()


CLV & Cohort Insights

High-CLV customers contribute disproportionate profit.

Retention drops sharply after first purchase.

Early engagement strategies can significantly improve lifetime value.